# Flash Sales Evaluation — Corrected Pipeline

Replaces the original Flash Sales CausalImpact analysis with methodological fixes:

| Original Flaw | Fix |
|---|---|
| 2-day post-period (daily) | Hourly granularity → 12 data points |
| 2-month gap pre↔post | Continuous 6-week pre-period |
| Christmas in pre-period | Holiday-free pre-period |
| All controls aggregated to 'X' | Individual control city time series |
| Hour filter commented out | Active hour filter (16-22) |
| No contamination check | OMT check on control cities |
| 8 segments, no correction | Benjamini-Hochberg FDR correction |

## 1. Configuration

In [ ]:
import pandas as pd
import numpy as np
import time, gc, warnings
warnings.filterwarnings('ignore', category=FutureWarning)

from google.colab import auth
from google.cloud import bigquery

auth.authenticate_user()

from google.colab import drive
drive.mount('/content/drive')

import sys, os
CONFIG_DIR = '/content/drive/MyDrive/aa_test'
sys.path.insert(0, CONFIG_DIR)
os.chdir(CONFIG_DIR)

import aa_config as cfg
import flash_sales_utils as fsu

client = bigquery.Client(project=cfg.PROJECT_ID)

# === CONFIGURE YOUR FLASH SALES HERE ===
TREATMENT_CITY = 'Essen'
COUNTRY = 'DE'
FLASH_SALES_DATES = [
    ('2026-02-27', '2026-02-28'),  # Essen dinner Flash Sales
]
HOUR_START = 16
HOUR_END = 22
PRE_WEEKS = 6  # continuous pre-period

# Compute pre-period from first Flash Sales date
first_post_start = pd.to_datetime(FLASH_SALES_DATES[0][0])
pre_end = (first_post_start - pd.Timedelta(days=1)).strftime('%Y-%m-%d')
pre_start = (first_post_start - pd.Timedelta(weeks=PRE_WEEKS)).strftime('%Y-%m-%d')
last_post_end = FLASH_SALES_DATES[-1][1]

print(f'Treatment: {TREATMENT_CITY}')
print(f'Flash Sales: {FLASH_SALES_DATES}')
print(f'Hours: {HOUR_START}:00 - {HOUR_END}:00')
print(f'Pre-period: {pre_start} to {pre_end}')

## 2. Control City Selection (MAVERICK)

In [ ]:
df_cities = fsu.get_city_features(client, COUNTRY, pre_end)
print(f'{len(df_cities)} cities with features')

controls = fsu.find_control_cities(
    df_cities, TREATMENT_CITY, cfg.FLASH_SALES_KNN_NEIGHBORS)
print(f'\nKNN candidates ({len(controls)}): {controls}')

## 3. Hourly Data & Correlation Filter

In [ ]:
all_cities = [TREATMENT_CITY] + controls
df_orders = fsu.get_hourly_orders(
    client, COUNTRY, all_cities,
    pre_start, last_post_end,
    HOUR_START, HOUR_END)
print(f'Hourly orders: {len(df_orders)} rows, '
      f'{df_orders["city"].nunique()} cities')

# Correlation filter on pre-period only
df_pre = df_orders[
    pd.to_datetime(df_orders['order_hour']) <
    pd.to_datetime(FLASH_SALES_DATES[0][0])
]
filtered_controls = fsu.apply_correlation_filter(
    df_pre, TREATMENT_CITY, controls,
    cfg.FLASH_SALES_CORRELATION_THRESHOLD)
print(f'\nFinal control cities ({len(filtered_controls)}): '
      f'{filtered_controls}')

## 4. Contamination Check

In [ ]:
contam = fsu.check_control_contamination(
    client, COUNTRY, filtered_controls,
    FLASH_SALES_DATES[0][0], FLASH_SALES_DATES[-1][1])

contaminated_cities = contam['city'].unique().tolist() if len(contam) > 0 else []
clean_controls = [c for c in filtered_controls if c not in contaminated_cities]
print(f'\nClean controls: {len(clean_controls)}/{len(filtered_controls)}')
print(f'Using: {clean_controls}')

## 5. CausalImpact — Total Level

In [ ]:
post_start = FLASH_SALES_DATES[0][0]
post_end = FLASH_SALES_DATES[-1][1]

pivot, pre_period, post_period = fsu.build_hourly_pivot(
    df_orders, TREATMENT_CITY, clean_controls,
    pre_start, pre_end, post_start, post_end)

result_total = fsu.run_causal_impact(pivot, pre_period, post_period)

if result_total:
    incr = result_total['actual_orders'] - result_total['predicted_orders']
    print(f'\n=== TOTAL LEVEL RESULTS ===')
    print(f'Actual orders:    {result_total["actual_orders"]:.0f}')
    print(f'Predicted orders: {result_total["predicted_orders"]:.0f}')
    print(f'Incremental:      {incr:+.0f}')
    print(f'Raw uplift:       {result_total["uplift"]:+.4f} '
          f'({result_total["uplift"]*100:+.1f}%)')
else:
    print('CausalImpact failed at total level.')

## 6. A/A Bias Correction

In [ ]:
# Load A/A results for Flash Sales BSTS
df_aa = client.query(f"""
    SELECT campaign_period_uplift
    FROM `{cfg.AA_RESULTS_TABLE}`
    WHERE technique = '{cfg.FLASH_SALES_TECHNIQUE}'
      AND campaign_period_uplift IS NOT NULL
""").to_dataframe()

if len(df_aa) > 0 and result_total:
    from scipy import stats

    aa_uplifts = df_aa['campaign_period_uplift'].values
    aa_mean = np.mean(aa_uplifts)
    aa_std = np.std(aa_uplifts, ddof=1)
    n_aa = len(aa_uplifts)
    aa_ci = stats.t.interval(0.95, df=n_aa-1,
                              loc=aa_mean,
                              scale=aa_std / np.sqrt(n_aa))

    corrected_uplift = result_total['uplift'] - aa_mean
    mde = fsu.compute_mde(aa_uplifts)

    print(f'\n=== BIAS CORRECTION ===')
    print(f'A/A bias: {aa_mean:+.4f} ({aa_mean*100:+.1f}%), '
          f'95% CI [{aa_ci[0]:+.4f}, {aa_ci[1]:+.4f}]')
    print(f'Raw uplift:       {result_total["uplift"]:+.4f}')
    print(f'Corrected uplift: {corrected_uplift:+.4f} '
          f'({corrected_uplift*100:+.1f}%)')
    print(f'MDE (80% power):  {mde:.4f} ({mde*100:.1f}%)')

    if abs(result_total['uplift']) < mde:
        print(f'\nWARNING: Raw effect ({abs(result_total["uplift"])*100:.1f}%) '
              f'is below MDE ({mde*100:.1f}%). '
              f'Result is directional only \u2014 insufficient power to confirm.')
    else:
        print(f'\nEffect ({abs(result_total["uplift"])*100:.1f}%) '
              f'exceeds MDE ({mde*100:.1f}%).')
else:
    print('No A/A results available for bias correction.')
    print('Run aa_flash_sales_city_lookalike.ipynb first.')

## 7. Per-Segment Analysis (with FDR Correction)

In [ ]:
# Query hourly orders with segment join
def get_hourly_orders_segmented(client, country, cities, start, end,
                                 hour_start, hour_end):
    city_list = ','.join([f"'{c}'" for c in cities])
    query = f"""
    SELECT
        TIMESTAMP(DATE(fo.orderdatetime),
                  MAKE_TIME(EXTRACT(HOUR FROM fo.orderdatetime), 0, 0)) AS order_hour,
        dr.city,
        COALESCE(s.base_segment, 'unknown') AS base_segment,
        SUM(fo.nroforders) AS totalorders
    FROM `just-data-warehouse.dwh.fact_order` AS fo
    INNER JOIN `just-data-warehouse.dwh.dim_restaurant` AS dr
        ON fo.restaurantid = dr.restaurantid
    LEFT JOIN `just-data-warehouse.core_dwh.dim_unique_customer_history` AS ch
        ON fo.customerid = ch.customer_id
        AND ch.snapshot_date = DATE(fo.orderdatetime)
    LEFT JOIN `just-data-warehouse.dwh.fact_segmentation_scv_key` AS s
        ON ch.scv_key = s.scv_key
        AND s.snapshot_date = DATE_SUB(DATE(fo.orderdatetime), INTERVAL 1 DAY)
    WHERE dr.country = '{country}'
        AND dr.city IN ({city_list})
        AND DATE(fo.orderdatetime) >= DATE('{start}')
        AND DATE(fo.orderdatetime) <= DATE('{end}')
        AND EXTRACT(HOUR FROM fo.orderdatetime) >= {hour_start}
        AND EXTRACT(HOUR FROM fo.orderdatetime) < {hour_end}
    GROUP BY 1, 2, 3
    ORDER BY 1, 2, 3
    """
    return client.query(query).to_dataframe()

all_cities_clean = [TREATMENT_CITY] + clean_controls
df_seg = get_hourly_orders_segmented(
    client, COUNTRY, all_cities_clean,
    pre_start, last_post_end, HOUR_START, HOUR_END)
print(f'Segmented hourly orders: {len(df_seg)} rows')

segment_results = []
for segment in cfg.BASE_SEGMENTS:
    if segment == 'unknown':
        continue
    df_s = df_seg[df_seg['base_segment'] == segment].copy()
    df_s = df_s.groupby(['order_hour', 'city'])['totalorders'].sum().reset_index()

    if df_s.empty or df_s['city'].nunique() < 2:
        continue

    try:
        pivot_s, pre_p, post_p = fsu.build_hourly_pivot(
            df_s, TREATMENT_CITY, clean_controls,
            pre_start, pre_end, post_start, post_end)

        result_s = fsu.run_causal_impact(pivot_s, pre_p, post_p)
        if result_s:
            segment_results.append({
                'segment': segment,
                'uplift': result_s['uplift'],
                'incr_orders': result_s['actual_orders'] - result_s['predicted_orders'],
                'p_value': result_s['p_value'],
            })
            print(f'  {segment}: uplift={result_s["uplift"]:+.4f}')
    except Exception as e:
        print(f'  {segment}: ERROR - {e}')
    gc.collect()

# FDR correction
if segment_results:
    from statsmodels.stats.multitest import multipletests
    df_seg_results = pd.DataFrame(segment_results)
    p_vals = df_seg_results['p_value'].fillna(1.0).values
    reject, pvals_corrected, _, _ = multipletests(p_vals, alpha=0.05, method='fdr_bh')
    df_seg_results['p_value_fdr'] = pvals_corrected
    df_seg_results['significant_fdr'] = reject
    print(f'\n=== PER-SEGMENT RESULTS (FDR corrected) ===')
    print(df_seg_results.to_string(index=False))

## 8. Summary

In [ ]:
print('=' * 60)
print('FLASH SALES EVALUATION SUMMARY')
print('=' * 60)
print(f'Treatment city:    {TREATMENT_CITY}')
print(f'Flash Sales dates: {FLASH_SALES_DATES}')
print(f'Hours:             {HOUR_START}:00 - {HOUR_END}:00')
print(f'Control cities:    {len(clean_controls)} '
      f'(contaminated excluded: {len(contaminated_cities)})')
print(f'Pre-period:        {pre_start} to {pre_end} '
      f'({PRE_WEEKS} weeks)')
print()

if result_total:
    print(f'Raw uplift:        {result_total["uplift"]:+.4f} '
          f'({result_total["uplift"]*100:+.1f}%)')
    incr = result_total['actual_orders'] - result_total['predicted_orders']
    print(f'Incremental orders:{incr:+.0f}')

if len(df_aa) > 0 and result_total:
    print(f'A/A bias:          {aa_mean:+.4f} ({aa_mean*100:+.1f}%)')
    print(f'Corrected uplift:  {corrected_uplift:+.4f} '
          f'({corrected_uplift*100:+.1f}%)')
    print(f'MDE (80% power):   {mde*100:.1f}%')
    detectable = abs(result_total['uplift']) >= mde
    print(f'Effect detectable: {"Yes" if detectable else "No (directional only)"}')

print()
if segment_results:
    n_sig = sum(1 for r in segment_results if r.get('significant_fdr', False))
    print(f'Segments significant (FDR): {n_sig}/{len(segment_results)}')
print('=' * 60)